# SHAP from Scratch — Exact Shapley Values on the Airfoil Self-Noise Dataset

This notebook builds **SHAP values from zero**, without using the `shap` library for the main calculation.  
At the end, we use the official SHAP library only to verify that our manual implementation is correct.

## What we are trying to understand

For one model prediction,

$$
f(x),
$$

SHAP asks:

> **How much did each feature contribute to moving the prediction away from a baseline prediction?**

The final decomposition is

$$
\boxed{f(x)=v(\emptyset)+\sum_{i=1}^{M}\phi_i}
$$

where:

- $f(x)$: prediction for the observation we want to explain,
- $v(\emptyset)$: baseline / expected prediction,
- $M$: number of input features,
- $\phi_i$: SHAP value of feature $i$.

A positive SHAP value pushes the prediction **above** the baseline.  
A negative SHAP value pushes it **below** the baseline.



## 1. Load the dataset

We use the **UCI Airfoil Self-Noise** regression dataset.

The five input features are:

1. `frequency`
2. `attack-angle`
3. `chord-length`
4. `free-stream-velocity`
5. `suction-side-displacement-thickness`

The target is the measured sound-pressure level.

Why is this dataset useful for learning exact SHAP?

$$
M=5
$$

so there are only

$$
\boxed{2^5=32}
$$

possible feature coalitions. That is small enough to enumerate exactly.

> If `ucimlrepo` is not installed, run: `pip install ucimlrepo`

In [2]:
from ucimlrepo import fetch_ucirepo

airfoil = fetch_ucirepo(id=291)

X = airfoil.data.features.copy()
y = airfoil.data.targets.squeeze()

print(X.head())
print("\nX shape:", X.shape)
print("y shape:", y.shape)

   frequency  attack-angle  chord-length  free-stream-velocity  \
0        800           0.0        0.3048                  71.3   
1       1000           0.0        0.3048                  71.3   
2       1250           0.0        0.3048                  71.3   
3       1600           0.0        0.3048                  71.3   
4       2000           0.0        0.3048                  71.3   

   suction-side-displacement-thickness  
0                             0.002663  
1                             0.002663  
2                             0.002663  
3                             0.002663  
4                             0.002663  

X shape: (1503, 5)
y shape: (1503,)


## 2. Train a nonlinear prediction model

We use a `RandomForestRegressor`.

SHAP does **not** explain the true physical process directly. It explains the behavior of the **trained model**:

$$
x \longrightarrow f(x)
$$

So our question is not:

> What physically causes noise?

Instead it is:

> What features does this random-forest model use to produce this particular prediction?

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=6,
    random_state=42
)

model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,6
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## 3. Select one observation to explain

SHAP can explain one observation at a time.

Let

$$
x=(x_1,x_2,\dots,x_M)
$$

be the test observation we want to explain.

The model gives

$$
f(x).
$$

Later we will decompose this prediction into

$$
f(x)
=
\text{baseline}
+
\text{frequency contribution}
+
\text{angle contribution}
+
\cdots
$$

In [4]:
x = X_test.iloc[0]

print("Observation to explain:")
print(x)

# Keep the feature names when predicting to avoid sklearn warnings.
x_frame = x.to_frame().T
prediction = model.predict(x_frame)[0]

print("\nModel prediction:", prediction)

Observation to explain:
frequency                              400.000000
attack-angle                             0.000000
chord-length                             0.304800
free-stream-velocity                    31.700000
suction-side-displacement-thickness      0.003313
Name: 51, dtype: float64

Model prediction: 125.30057104464399


## 4. Choose the background dataset

This is one of the most important ideas in SHAP.

The model expects all five features. If we say that only one feature is "known", we cannot literally delete the other columns.

Instead, our manual implementation represents the unknown features using rows from a **background dataset**.

We use 20 training observations:

$$
B=20.
$$

For a coalition $S$ of known features:

- features inside $S$ are fixed to the values from the observation $x$,
- features outside $S$ come from each background row,
- we predict every resulting row,
- then average the predictions.

This notebook therefore implements the same basic masking idea as **interventional / independent-background SHAP**.

The background is part of the explanation. A different background distribution can produce different SHAP values.

In [5]:
background = X_train.iloc[:20].copy()

print("Background shape:", background.shape)
display(background.head())

Background shape: (20, 5)


,frequency,attack-angle,chord-length,free-stream-velocity,suction-side-displacement-thickness
799,1000,8.4,0.0508,71.3,0.005295
1349,2000,6.7,0.1016,71.3,0.004783
1428,400,12.3,0.1016,55.5,0.036823
1150,400,17.4,0.0254,71.3,0.016104
1100,400,9.5,0.0254,31.7,0.004614


## 5. Baseline: $v(\emptyset)$

First pretend that **none** of the features of the target observation are known.

The empty coalition is

$$
S=\emptyset.
$$

We evaluate the model on every background row and average:

$$
\boxed{v(\emptyset) = \frac{1}{B} \sum_{b=1}^{B} f(x^{(b)})}
$$

where $x^{(b)}$ is background row $b$.

Interpretation:

> Before we know anything specific about the observation we are explaining, what prediction does the model produce on average over the background data?

This is the starting point from which SHAP feature contributions push the prediction upward or downward.

In [6]:
baseline = model.predict(background).mean()

print("Baseline v(empty set):", baseline)

Baseline v(empty set): 124.70042988800044


## 6. What does "velocity only" mean?

Suppose the selected observation has

$$
x_{\text{velocity}}=31.7.
$$

"Velocity only" means:

- keep `free-stream-velocity = 31.7` for every masked row,
- take all other feature values from the background rows.

For background row $b$, the masked observation is conceptually

$$
z^{(b)}
=
(
x^{(b)}_{\text{frequency}},
x^{(b)}_{\text{angle}},
x^{(b)}_{\text{chord}},
x_{\text{velocity}},
x^{(b)}_{\text{thickness}}
).
$$

Then

$$
\boxed{v(\{\text{velocity}\}) = \frac{1}{B} \sum_{b=1}^{B} f(z^{(b)})}
$$

So "velocity only" does **not** mean that the random forest becomes a one-feature model.  
It means we know the target observation's velocity and average over background values for everything else.

In [7]:
velocity_only = background.copy()

velocity_only["free-stream-velocity"] = x["free-stream-velocity"]

velocity_only_predictions = model.predict(velocity_only)
v_velocity = velocity_only_predictions.mean()

print("v({velocity}):", v_velocity)
print(
    "Contribution when velocity enters first:",
    v_velocity - baseline
)

v({velocity}): 124.05696051050101
Contribution when velocity enters first: -0.6434693774994287


## 7. General coalition value $v(S)$

Now generalize the previous idea.

Let $F=\{1,\dots,M\}$ be the set of all features, and let

$$
S\subseteq F
$$

be a coalition of features that are considered known.

For each background row $b$, create a masked row

$$
z_j^{(b,S)}
=
\begin{cases}
x_j, & j\in S,\\
x_j^{(b)}, & j\notin S.
\end{cases}
$$

Then define

$$
\boxed{v(S) = \frac{1}{B} \sum_{b=1}^{B} f\!\left(z^{(b,S)}\right)}
$$

Interpretation:

> What does the model predict on average when we know the features in $S$ from the target observation and treat all other features as unknown/background?

Examples:

$$
v(\emptyset)
$$

means no target features are known.

$$
v(\{\text{velocity}\})
$$

means only velocity is known.

$$
v(\{\text{frequency},\text{velocity}\})
$$

means frequency and velocity are known.

$$
v(F)=f(x)
$$

means all features are known, so the masked rows become the actual target observation.

In [8]:
import numpy as np

def coalition_value(model, x, background, coalition):
    """
    Compute v(S) for an independent-background coalition.

    Parameters
    ----------
    model : fitted model
    x : pandas Series
        Observation being explained.
    background : pandas DataFrame
        Rows used to fill features that are not in the coalition.
    coalition : iterable of int
        Feature indexes that are considered known.

    Returns
    -------
    float
        Average model prediction over the masked background rows.
    """
    masked = background.copy()

    for feature_idx in coalition:
        feature_name = background.columns[feature_idx]
        masked[feature_name] = x.iloc[feature_idx]

    return model.predict(masked).mean()

### Check a few coalitions

Our feature indexes are:

$$
0=\text{frequency},\quad
1=\text{attack angle},\quad
2=\text{chord length},\quad
3=\text{velocity},\quad
4=\text{thickness}.
$$

Therefore:

- `[]` means $v(\emptyset)$,
- `[3]` means $v(\{\text{velocity}\})$,
- `[0, 3]` means $v(\{\text{frequency},\text{velocity}\})$.

In [ ]:
print("v(empty):", coalition_value(model, x, background, []))
print("v({velocity}):", coalition_value(model, x, background, [3]))
print(
    "v({frequency, velocity}):",
    coalition_value(model, x, background, [0, 3])
)

## 8. Marginal contribution of one feature

A feature's contribution depends on **what is already known**.

For feature $i$ and coalition $S$ that does not contain $i$, define the marginal contribution

$$
\boxed{\Delta_i(S) = v(S\cup\{i\})-v(S)}
$$

Example for velocity:

$$
\Delta_{\text{velocity}}(\emptyset)
=
v(\{\text{velocity}\})-v(\emptyset).
$$

But velocity might contribute differently when frequency is already known:

$$
\Delta_{\text{velocity}}(\{\text{frequency}\})
=
v(\{\text{frequency},\text{velocity}\})
-
v(\{\text{frequency}\}).
$$

This is why SHAP does not use only one "feature removed" calculation.  
It measures the feature in **every possible context**.

## 9. Why do Shapley values need weights?

With $M=5$ features, fix one feature $i$. The other four features can form

$$
\boxed{2^{M-1}=2^4=16}
$$

coalitions that do not contain $i$.

However, simply averaging those 16 coalition differences would **not** reproduce the classical Shapley value.

Shapley values are equivalent to:

> Put the features in every possible arrival order and average the contribution of feature $i$ at the moment it arrives.

There are

$$
M!
$$

possible feature orderings.

For a coalition $S$ of size $|S|$, the number of orders in which:

- all members of $S$ appear before feature $i$, and
- all remaining features appear after $i$,

is

$$
|S|!(M-|S|-1)!.
$$

Therefore the probability/weight of that coalition context is

$$
\boxed{w(S) = \frac{|S|!(M-|S|-1)!}{M!}}
$$

For $M=5$, the weight depends only on coalition size:

| $|S|$ | Weight |
|---:|---:|
| 0 | $1/5$ |
| 1 | $1/20$ |
| 2 | $1/30$ |
| 3 | $1/20$ |
| 4 | $1/5$ |

These weights ensure that every possible **feature arrival order** is treated fairly.

## 10. Exact Shapley / SHAP formula

For feature $i$,

$$
\boxed{\phi_i = \sum_{S\subseteq F\setminus\{i\}} \frac{|S|!(M-|S|-1)!}{M!} \left[ v(S\cup\{i\})-v(S) \right]}
$$

Read the formula from the inside outward:

### A. Choose a coalition
$$
S\subseteq F\setminus\{i\}.
$$

### B. Measure what feature $i$ adds
$$
v(S\cup\{i\})-v(S).
$$

### C. Weight this context fairly
$$
\frac{|S|!(M-|S|-1)!}{M!}.
$$

### D. Sum over every coalition that does not already contain $i$

The result is the exact SHAP value

$$
\phi_i.
$$

In [9]:
from itertools import combinations
from math import factorial

def exact_shap_feature(model, x, background, feature_i):
    """Compute the exact Shapley value phi_i for one feature."""
    M = len(x)

    all_features = list(range(M))
    other_features = [
        j for j in all_features
        if j != feature_i
    ]

    phi_i = 0.0

    # S may contain 0, 1, ..., M-1 of the OTHER features.
    for coalition_size in range(M):

        for coalition_tuple in combinations(
            other_features,
            coalition_size
        ):
            S = list(coalition_tuple)

            # v(S)
            v_without_i = coalition_value(
                model,
                x,
                background,
                S
            )

            # v(S union {i})
            S_with_i = S + [feature_i]
            v_with_i = coalition_value(
                model,
                x,
                background,
                S_with_i
            )

            # Delta_i(S)
            marginal_contribution = (
                v_with_i - v_without_i
            )

            # w(S)
            weight = (
                factorial(coalition_size)
                * factorial(M - coalition_size - 1)
                / factorial(M)
            )

            phi_i += weight * marginal_contribution

    return phi_i

## 11. Compute exact SHAP values for all five features

We run the previous function once for each feature:

$$
\phi_1,\phi_2,\ldots,\phi_5.
$$

The sign tells us the direction:

- $\phi_i>0$: feature pushes the prediction upward from the baseline,
- $\phi_i<0$: feature pushes it downward.

The magnitude

$$
|\phi_i|
$$

tells us how strong that feature's contribution is for **this observation**.

In [10]:
manual_shap = np.array([
    exact_shap_feature(
        model,
        x,
        background,
        feature_i=i
    )
    for i in range(X.shape[1])
])

manual_results = X.columns.to_frame(name="feature")
manual_results["feature_value"] = x.values
manual_results["manual_SHAP"] = manual_shap

display(manual_results)

,feature,feature_value,manual_SHAP
frequency,frequency,400.000000,0.900145
attack-angle,attack-angle,0.000000,-0.923801
chord-length,chord-length,0.304800,0.278506
free-stream-velocity,free-stream-velocity,31.700000,-0.571105
suction-side-displacement-thickness,suction-side-displacement-thickness,0.003313,0.916396


## 12. Local accuracy / efficiency property

A central Shapley property is that the feature contributions exactly explain the difference between the baseline and the prediction:

$$
\boxed{\sum_{i=1}^{M}\phi_i = f(x)-v(\emptyset)}
$$

Therefore,

$$
\boxed{f(x) = v(\emptyset) + \sum_{i=1}^{M}\phi_i}
$$

This is called **efficiency** in cooperative game theory and is closely related to **local accuracy** in SHAP explanations.

If our implementation is correct, reconstructing the prediction this way should reproduce the random-forest prediction up to floating-point precision.

In [11]:
baseline = model.predict(background).mean()
prediction = model.predict(x_frame)[0]
reconstructed = baseline + manual_shap.sum()

print("Baseline:             ", baseline)
print("Sum of SHAP values:   ", manual_shap.sum())
print("Reconstructed output: ", reconstructed)
print("Model prediction:     ", prediction)
print("Absolute error:       ", abs(reconstructed - prediction))

assert np.isclose(reconstructed, prediction, atol=1e-10)

Baseline:              124.70042988800044
Sum of SHAP values:    0.6001411566435737
Reconstructed output:  125.30057104464402
Model prediction:      125.30057104464399
Absolute error:        2.842170943040401e-14


## 13. Optional: inspect all coalition values

Because this dataset has only five features, there are only

$$
2^5=32
$$

possible coalitions.

Printing all $v(S)$ values is useful for learning because it lets you see the exact numbers from which SHAP is constructed.

The full coalition $F$ should satisfy

$$
v(F)=f(x),
$$

and the empty coalition should satisfy

$$
v(\emptyset)=\text{baseline}.
$$

In [12]:
import pandas as pd

feature_names = list(X.columns)
all_coalition_rows = []

for size in range(len(feature_names) + 1):
    for coalition_tuple in combinations(
        range(len(feature_names)),
        size
    ):
        coalition = list(coalition_tuple)
        names = [
            feature_names[i]
            for i in coalition
        ]

        all_coalition_rows.append({
            "coalition_indices": tuple(coalition),
            "coalition_features": names if names else ["EMPTY"],
            "size": size,
            "v(S)": coalition_value(
                model,
                x,
                background,
                coalition
            ),
        })

coalition_table = pd.DataFrame(all_coalition_rows)
display(coalition_table)

,coalition_indices,coalition_features,size,v(S)
0,(),[EMPTY],0,124.700430
1,"(0,)",[frequency],1,125.028379
2,"(1,)",[attack-angle],1,124.746019
3,"(2,)",[chord-length],1,124.766060
4,"(3,)",[free-stream-velocity],1,124.056961
5,"(4,)",[suction-side-displacement-thickness],1,125.834928
6,"(0, 1)","[frequency, attack-angle]",2,124.830713
7,"(0, 2)","[frequency, chord-length]",2,126.904241
8,"(0, 3)","[frequency, free-stream-velocity]",2,124.312641
9,"(0, 4)","[frequency, suction-side-displacement-thickness]",2,125.920663


## 14. Compare our implementation with the SHAP library

Only now do we import `shap`.

To make the comparison mathematically fair, both implementations must use:

1. the **same model**,
2. the **same target observation**,
3. the **same background rows**,
4. the **same independent masking definition**.

We use

```python
shap.maskers.Independent(background)
```

because it matches the masking strategy used in our manual `coalition_value` function.

Then we request the library's exact explainer.

> If SHAP is not installed, run: `pip install shap`

In [13]:
import shap

masker = shap.maskers.Independent(
    background,
    max_samples=len(background)
)

explainer = shap.Explainer(
    model.predict,
    masker,
    algorithm="exact"
)

library_result = explainer(x_frame)
library_shap = library_result.values[0]

c:\Users\Nicki\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 15. Numerical comparison

For every feature we compare

$$
\phi_i^{\text{manual}}
$$

with

$$
\phi_i^{\text{SHAP library}}.
$$

The difference should be approximately zero:

$$
\boxed{\phi_i^{\text{manual}} - \phi_i^{\text{library}} \approx0}
$$

Small differences on the order of $10^{-12}$ or $10^{-15}$ can occur because computers use finite-precision floating-point arithmetic.

In [14]:
comparison = pd.DataFrame({
    "feature": X.columns,
    "feature_value": x.values,
    "manual_SHAP": manual_shap,
    "SHAP_library": library_shap,
})

comparison["difference"] = (
    comparison["manual_SHAP"]
    - comparison["SHAP_library"]
)

display(comparison)

print(
    "All values match:",
    np.allclose(
        manual_shap,
        library_shap,
        atol=1e-8
    )
)

,feature,feature_value,manual_SHAP,SHAP_library,difference
0,frequency,400.000000,0.900145,0.900145,2.642331e-14
1,attack-angle,0.000000,-0.923801,-0.923801,2.886580e-15
2,chord-length,0.304800,0.278506,0.278506,0.000000e+00
3,free-stream-velocity,31.700000,-0.571105,-0.571105,1.243450e-14
4,suction-side-displacement-thickness,0.003313,0.916396,0.916396,5.551115e-15


All values match: True


## 16. What this notebook actually implemented

The calculation pipeline is:

$$
\boxed{\text{background data} \rightarrow v(S) \rightarrow \Delta_i(S) \rightarrow w(S) \rightarrow \phi_i}
$$

### Baseline

$$
v(\emptyset)
=
E[f(X_{\text{background}})].
$$

### Coalition value

Under the independent-background masking used here,

$$
v(S)
=
\frac{1}{B}\sum_{b=1}^{B}
f(z^{(b,S)}).
$$

### Marginal contribution

$$
\Delta_i(S)
=
v(S\cup\{i\})-v(S).
$$

### Shapley weight

$$
w(S)
=
\frac{|S|!(M-|S|-1)!}{M!}.
$$

### Feature attribution

$$
\phi_i
=
\sum_S w(S)\Delta_i(S).
$$

### Prediction reconstruction

$$
f(x)
=
v(\emptyset)
+
\sum_i\phi_i.
$$

---

## Important interpretation warning

A SHAP value explains the **model**, not causality.

If

$$
\phi_{\text{velocity}}<0,
$$

the correct statement is:

> For this observation and this background/reference distribution, velocity pushed the model prediction below the baseline.

It does **not** automatically mean:

> Increasing velocity physically causes the sound level to decrease.

Also, correlated features require care. Independent masking can create feature combinations that are uncommon or physically unrealistic. Conditional SHAP uses a different definition of $v(S)$ intended to respect dependencies between features.

---

## Final mental model

Imagine each feature is a player joining a team.

For feature $i$:

1. consider every possible group of features that could arrive before it,
2. measure how much the prediction changes when $i$ joins,
3. weight each context according to how often it occurs among all feature orderings,
4. average those contributions.

That fair average is the feature's **Shapley / SHAP value**.